<a href="https://colab.research.google.com/github/Mohamed-Shawky281/Hybrid-RAG-Research-assistant/blob/main/Hybrid_RAG_Research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hybrid RAG - Research Assistant Project

##Downloading libraries and dependencies...

In [3]:
!pip install -q langchain langchain-community langchain-chroma chromadb pypdf sentence-transformers rank_bm25

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/rag_research_assistant"
os.makedirs(f"{PROJECT_DIR}/papers", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/processed", exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)

Mounted at /content/drive
Project folder ready at: /content/drive/MyDrive/rag_research_assistant


In [5]:
from pathlib import Path

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s) found in papers/:")
for p in pdf_paths:
    print(" -", p.name)

10 PDF(s) found in papers/:
 - CryptographyinPostQuantumComputingEra (1).pdf
 - DOC-20260822-WA0006_260915_154729 (1).pdf
 - DOC-20260827-WA0014_260915_154811 (1).pdf
 - DOC-20260827-WA0015_260915_155007 (1).pdf
 - DOC-20260827-WA0016_260915_155026 (1).pdf
 - CryptographyinPostQuantumComputingEra.pdf
 - DOC-20260827-WA0014_260915_154811.pdf
 - DOC-20260827-WA0016_260915_155026.pdf
 - DOC-20260822-WA0006_260915_154729.pdf
 - DOC-20260827-WA0015_260915_155007.pdf


Uploading The wanted reference documents

In [6]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded.keys():
    dest = f"{PROJECT_DIR}/papers/{fname}"
    with open(dest, "wb") as f:
        f.write(uploaded[fname])
    print(f"Saved {fname} -> {dest}")

Saving CryptographyinPostQuantumComputingEra.pdf to CryptographyinPostQuantumComputingEra.pdf
Saving DOC-20260822-WA0006_260915_154729.pdf to DOC-20260822-WA0006_260915_154729.pdf
Saving DOC-20260827-WA0014_260915_154811.pdf to DOC-20260827-WA0014_260915_154811.pdf
Saving DOC-20260827-WA0015_260915_155007.pdf to DOC-20260827-WA0015_260915_155007.pdf
Saving DOC-20260827-WA0016_260915_155026.pdf to DOC-20260827-WA0016_260915_155026.pdf
Saved CryptographyinPostQuantumComputingEra.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf
Saved DOC-20260822-WA0006_260915_154729.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260822-WA0006_260915_154729.pdf
Saved DOC-20260827-WA0014_260915_154811.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0014_260915_154811.pdf
Saved DOC-20260827-WA0015_260915_155007.pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0015_260915_155007.pdf
Sa

##Loading PDFs into LangChain Documents

In [7]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))

all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(str(path))
    docs = loader.load()  # one Document per page, metadata already includes 'source' and 'page'
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages from {len(pdf_paths)} papers")
print(all_docs[0].page_content[:500])
print(all_docs[0].metadata)

/tmp/ipykernel_1328/455741346.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 270 pages from 10 papers
1 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
 
 
 
Cryptography in Post Quantum Computing Era 
Neerav Sood 
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


##Chunking (including overlapping)

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # characters -- LangChain's default unit
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph breaks first, falls back to smaller units
)

split_docs = splitter.split_documents(all_docs)
print(f"Created {len(split_docs)} chunks from {len(all_docs)} pages")
print(split_docs[0].page_content[:300])
print(split_docs[0].metadata)

Created 920 chunks from 270 pages
1 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
 
 
 
Cryptography in Post Quantum Computing Era 
Neerav Sood 
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


##Building the Chroma vector store (Meaning Similarity)

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=f"{PROJECT_DIR}/chroma_db",
)

print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

/tmp/ipykernel_1328/2980357329.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma vector store built with 2760 chunks


In [10]:
print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

Chroma vector store built with 2760 chunks


In [11]:
import pickle
import os

file_path = f"{PROJECT_DIR}/processed/split_docs.pkl"

# Check if the file exists. If not, and split_docs is available, save it.
# This assumes split_docs is defined in the current kernel state from previous cells.
if not os.path.exists(file_path):
    print(f"File '{file_path}' not found. Saving 'split_docs' to file first.")
    with open(file_path, "wb") as f:
        pickle.dump(split_docs, f)

# Now, attempt to load the file (which should now exist or existed already)
with open(file_path, "rb") as f:
    split_docs = pickle.load(f)

print(f"Loaded {len(split_docs)} chunks")

Loaded 920 chunks


##Build the Keyword similarity (BM25 retriever)

In [12]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 5  # how many chunks BM25 returns per query

print("BM25 retriever built")

BM25 retriever built
